# NLP

NLP, ou Processamento de Linguagem Natural, e uma area da ciencia de dados que estuda como transformar textos em informacao estruturada. Com essas tecnicas, conseguimos analisar noticias, identificar temas, contar palavras, comparar documentos e preparar dados textuais para modelos de aprendizado de maquina.

In [81]:
import json
from pathlib import Path

import pandas as pd

noticias = []

for arquivo in sorted(Path("../webscraping/data/").glob("*.json")):
    with arquivo.open(encoding="utf-8") as f:
        noticias.append(json.load(f))

df = pd.DataFrame(noticias)

print(f"Antes: {len(df)} noticias")

df = df.drop_duplicates(subset="url").reset_index(drop=True)

print(f"Depois: {len(df)} noticias")

df.head()

Antes: 27 noticias
Depois: 16 noticias


,url,titulo,descricao,tags,texto,data
0,https://www12.senado.leg.br/noticias/materias/...,Chico Rodrigues destaca ações em Roraima e cri...,"O senador Chico Rodrigues (PSB-RR), em pronunc...","[Agricultura familiar , Emendas parlamentares,...","O senador Chico Rodrigues (PSB-RR), em pronunc...",2026-03-30T17:31:28-03:00
1,https://www12.senado.leg.br/noticias/materias/...,"Com mais de 4 mil páginas, relatório da CPMI p...","Após sete meses de funcionamento, a CPMI do IN...","[Aposentados, INSS, Pensionistas]","Após sete meses de funcionamento, a CPMI do IN...",2026-03-27T19:39:12-03:00
2,https://www12.senado.leg.br/noticias/materias/...,Wilder defende prisão domiciliar para Jair Bol...,"O senador Wilder Morais (PL-GO), em pronunciam...","[Alexandre de Moraes, Jair Bolsonaro, Procurad...","O senador Wilder Morais (PL-GO), em pronunciam...",2026-03-24T16:59:17-03:00
3,https://www12.senado.leg.br/noticias/materias/...,CPMI do INSS: relatório será lido e pode ser v...,Após o Supremo Tribunal Federal (STF) rejeitar...,"[André Mendonça, Aposentados, Corrupção, INSS,...",Após o Supremo Tribunal Federal (STF) rejeitar...,2026-03-26T21:14:18-03:00
4,https://www12.senado.leg.br/noticias/materias/...,CPMI do INSS aprova mais uma convocação e queb...,Ainda sem definição quanto à sua prorrogação —...,"[Aposentados, Conselho de Controle de Atividad...",Ainda sem definição quanto à sua prorrogação —...,2026-03-26T20:05:27-03:00


In [82]:
df["texto"].iloc[0]

'O senador Chico Rodrigues (PSB-RR), em pronunciamento no Plenário nesta segunda-feira (30), destacou a execução de emendas parlamentares no estado de Roraima, com foco no apoio a produtores rurais. O parlamentar mencionou ações no município de Bonfim, onde acompanhou a entrega de equipamentos, e afirmou que os investimentos têm contribuído para fortalecer o setor.\n\n\nO senador ressaltou a entrega de tratores, caminhões e outros implementos ao longo do mandato, voltados principalmente ao atendimento de pequenos produtores em diferentes municípios de Roraima. Segundo ele, os recursos têm ampliado a capacidade produtiva da agricultura familiar e contribuído para suprir necessidades locais.\n\n\n— \nA agricultura familiar deu um grande salto a partir das nossas ações. Já entreguei mais de cem tratores agrícolas nesses sete anos de mandato de senador da República, atendendo os pequenos produtores rurais da agricultura familiar, com caminhões também, com equipamentos na área da agricultur

## Trabalhando apenas com o texto

Por enquanto, vamos trabalhar somente com a coluna `texto`, que contem o corpo completo de cada noticia. As outras colunas, como `titulo`, `descricao`, `tags` e `data`, continuam no `DataFrame`, mas vamos deixar elas de lado neste primeiro momento para entender melhor o conteudo textual.

## Passos da analise

Vamos preparar os textos aos poucos:

1. Limpar os textos.
2. Remover palavras muito comuns.
3. Criar uma representacao Bag of Words.
4. Contar palavras frequentes.
5. Transformar textos em numeros para analises posteriores.

## 1. Limpeza basica

Na celula abaixo:

- `wordpunct_tokenize` separa o texto em palavras e pontuacao.
- `texto.lower()` coloca tudo em minusculas.
- `unidecode(texto)` troca letras acentuadas por letras sem acento.
- `token.isalnum()` mantem apenas letras e numeros.
- `" ".join(tokens)` junta os tokens em uma frase limpa.
- `.apply(limpar_texto)` aplica a funcao em todas as noticias.

Exemplo: `"Olá, Senado!"` vira `"ola senado"`.

In [83]:
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode

In [84]:
texto_exemplo = df["texto"].iloc[0].lower()
texto_exemplo = unidecode(texto_exemplo)
texto_exemplo = wordpunct_tokenize(texto_exemplo)
texto_exemplo

['o',
 'senador',
 'chico',
 'rodrigues',
 '(',
 'psb',
 '-',
 'rr',
 '),',
 'em',
 'pronunciamento',
 'no',
 'plenario',
 'nesta',
 'segunda',
 '-',
 'feira',
 '(',
 '30',
 '),',
 'destacou',
 'a',
 'execucao',
 'de',
 'emendas',
 'parlamentares',
 'no',
 'estado',
 'de',
 'roraima',
 ',',
 'com',
 'foco',
 'no',
 'apoio',
 'a',
 'produtores',
 'rurais',
 '.',
 'o',
 'parlamentar',
 'mencionou',
 'acoes',
 'no',
 'municipio',
 'de',
 'bonfim',
 ',',
 'onde',
 'acompanhou',
 'a',
 'entrega',
 'de',
 'equipamentos',
 ',',
 'e',
 'afirmou',
 'que',
 'os',
 'investimentos',
 'tem',
 'contribuido',
 'para',
 'fortalecer',
 'o',
 'setor',
 '.',
 'o',
 'senador',
 'ressaltou',
 'a',
 'entrega',
 'de',
 'tratores',
 ',',
 'caminhoes',
 'e',
 'outros',
 'implementos',
 'ao',
 'longo',
 'do',
 'mandato',
 ',',
 'voltados',
 'principalmente',
 'ao',
 'atendimento',
 'de',
 'pequenos',
 'produtores',
 'em',
 'diferentes',
 'municipios',
 'de',
 'roraima',
 '.',
 'segundo',
 'ele',
 ',',
 'os',
 '

In [85]:



def limpar_texto(texto):
    texto = texto.lower()
    texto = unidecode(texto)
    tokens = wordpunct_tokenize(texto)
    tokens = [token for token in tokens if token.isalnum()]
    return " ".join(tokens)


df["texto_limpo"] = df["texto"].apply(limpar_texto)

df[["texto", "texto_limpo"]].iloc[0]

texto          O senador Chico Rodrigues (PSB-RR), em pronunc...
texto_limpo    o senador chico rodrigues psb rr em pronunciam...
Name: 0, dtype: str

## 2. Removendo stopwords

Stopwords sao palavras muito comuns, como `a`, `o`, `de`, `para` e `que`. Elas aparecem muito, mas geralmente ajudam pouco a entender o tema de um texto.

Na celula abaixo:

- `stopwords.words("portuguese")` carrega stopwords em portugues.
- `texto.split()` separa o texto limpo em palavras.
- `token not in stopwords_pt` remove as palavras muito comuns.
- `.str.join(" ")` junta os tokens restantes em um texto sem stopwords.
- `.apply(remover_stopwords)` aplica a funcao em todas as noticias.

Exemplo: `"o senador falou com a imprensa"` vira `["senador", "falou", "imprensa"]`.

In [86]:
import nltk

from nltk.corpus import stopwords


nltk.download("stopwords", quiet=True)

stopwords_pt = stopwords.words("portuguese")
stopwords_pt = [unidecode(palavra) for palavra in stopwords_pt]
stopwords_pt = set(stopwords_pt)

In [87]:
stopwords_pt

{'a',
 'ao',
 'aos',
 'aquela',
 'aquelas',
 'aquele',
 'aqueles',
 'aquilo',
 'as',
 'ate',
 'com',
 'como',
 'da',
 'das',
 'de',
 'dela',
 'delas',
 'dele',
 'deles',
 'depois',
 'do',
 'dos',
 'e',
 'ela',
 'elas',
 'ele',
 'eles',
 'em',
 'entre',
 'era',
 'eram',
 'eramos',
 'essa',
 'essas',
 'esse',
 'esses',
 'esta',
 'estamos',
 'estao',
 'estar',
 'estas',
 'estava',
 'estavam',
 'estavamos',
 'este',
 'esteja',
 'estejam',
 'estejamos',
 'estes',
 'esteve',
 'estive',
 'estivemos',
 'estiver',
 'estivera',
 'estiveram',
 'estiveramos',
 'estiverem',
 'estivermos',
 'estivesse',
 'estivessem',
 'estivessemos',
 'estou',
 'eu',
 'foi',
 'fomos',
 'for',
 'fora',
 'foram',
 'foramos',
 'forem',
 'formos',
 'fosse',
 'fossem',
 'fossemos',
 'fui',
 'ha',
 'haja',
 'hajam',
 'hajamos',
 'hao',
 'havemos',
 'haver',
 'hei',
 'houve',
 'houvemos',
 'houver',
 'houvera',
 'houveram',
 'houveramos',
 'houverao',
 'houverei',
 'houverem',
 'houveremos',
 'houveria',
 'houveriam',
 'h

In [88]:
"_".join(df["texto_limpo"].iloc[0].split())

'o_senador_chico_rodrigues_psb_rr_em_pronunciamento_no_plenario_nesta_segunda_feira_30_destacou_a_execucao_de_emendas_parlamentares_no_estado_de_roraima_com_foco_no_apoio_a_produtores_rurais_o_parlamentar_mencionou_acoes_no_municipio_de_bonfim_onde_acompanhou_a_entrega_de_equipamentos_e_afirmou_que_os_investimentos_tem_contribuido_para_fortalecer_o_setor_o_senador_ressaltou_a_entrega_de_tratores_caminhoes_e_outros_implementos_ao_longo_do_mandato_voltados_principalmente_ao_atendimento_de_pequenos_produtores_em_diferentes_municipios_de_roraima_segundo_ele_os_recursos_tem_ampliado_a_capacidade_produtiva_da_agricultura_familiar_e_contribuido_para_suprir_necessidades_locais_a_agricultura_familiar_deu_um_grande_salto_a_partir_das_nossas_acoes_ja_entreguei_mais_de_cem_tratores_agricolas_nesses_sete_anos_de_mandato_de_senador_da_republica_atendendo_os_pequenos_produtores_rurais_da_agricultura_familiar_com_caminhoes_tambem_com_equipamentos_na_area_da_agricultura_arados_grades_carretas_colheitad

In [89]:
def remover_stopwords(texto):
    tokens = texto.split()
    tokens = [token for token in tokens if token not in stopwords_pt]
    return tokens


df["tokens_sem_stopwords"] = df["texto_limpo"].apply(remover_stopwords)
df["texto_sem_stopwords"] = df["tokens_sem_stopwords"].str.join(" ")

df[["texto_limpo", "tokens_sem_stopwords", "texto_sem_stopwords"]].head()

,texto_limpo,tokens_sem_stopwords,texto_sem_stopwords
0,o senador chico rodrigues psb rr em pronunciam...,"[senador, chico, rodrigues, psb, rr, pronuncia...",senador chico rodrigues psb rr pronunciamento ...
1,apos sete meses de funcionamento a cpmi do ins...,"[apos, sete, meses, funcionamento, cpmi, inss,...",apos sete meses funcionamento cpmi inss inicio...
2,o senador wilder morais pl go em pronunciament...,"[senador, wilder, morais, pl, go, pronunciamen...",senador wilder morais pl go pronunciamento ple...
3,apos o supremo tribunal federal stf rejeitar a...,"[apos, supremo, tribunal, federal, stf, rejeit...",apos supremo tribunal federal stf rejeitar pro...
4,ainda sem definicao quanto a sua prorrogacao o...,"[ainda, definicao, quanto, prorrogacao, suprem...",ainda definicao quanto prorrogacao supremo tri...


## 3. Bag of Words

No Bag of Words, cada linha representa uma noticia e cada coluna representa uma palavra. O valor indica quantas vezes aquela palavra apareceu na noticia.

Exemplo: `"senador falou senador"` teria `senador = 2` e `falou = 1`.

In [90]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
matriz_bow = vectorizer.fit_transform(df["texto_sem_stopwords"])

df_bow = pd.DataFrame(
    matriz_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)

df_bow.head()

,02,04,061,0800,09,11,12,120,16,16h,...,votarmos,votos,voz,vulnerabilidades,weverton,wilder,wolney,xingamentos,zequinha,zettel
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,1,0,0,2,1,1,0,2,0,...,1,1,2,0,5,0,0,1,1,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,4


In [91]:
df["texto"].iloc[1]

'Após sete meses de funcionamento, a CPMI do INSS iniciou a análise de seu relatório final nesta sexta-feira (27). Com cerca de 4 mil páginas, o texto do relator, deputado Alfredo Gaspar (União-AL), propõe o indiciamento de 216 pessoas.\n\n\nEntre elas, parlamentares, ex-ministros, dirigentes e ex‑dirigentes do INSS e da Dataprev, além do banqueiro Daniel Vorcaro e do empresário Fábio Luís Lula da Silva, conhecido como Lulinha, filho do presidente da República, Luiz Inácio Lula da Silva. O relator pede ainda o indiciamento do ex-ministro do Trabalho e Previdência José Carlos Oliveira, que tem hoje o nome de Ahmed Mohamad Oliveira e\xa0comandou a pasta no governo Jair Bolsonaro, e do ex-ministro da Previdência Carlos Lupi, da atual gestão. Também estão citados a deputada Gorete Pereira (MDB-CE), o ex-deputado Euclydes Pettersen (Republicanos-MG) e o senador Weverton (PDT-MA).\n\n\nA versão lida por Alfredo Gaspar precisará ser votada pela CPMI, que vai decidir se o aprova ou não.\xa0O r

### Removendo colunas com numeros

Agora vamos remover qualquer coluna cujo nome tenha pelo menos um numero.

In [92]:
colunas_com_numeros = [col for col in df_bow.columns if any(char.isdigit() for char in col)]

df_bow = df_bow.drop(columns=colunas_com_numeros)

print(f"{len(colunas_com_numeros)} colunas removidas")
df_bow.head()

46 colunas removidas


,abertura,abordadas,abraao,abril,abrir,absolvicao,abuso,ac,acabou,acao,...,votarmos,votos,voz,vulnerabilidades,weverton,wilder,wolney,xingamentos,zequinha,zettel
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,0,0,0,1,0,0,...,1,1,2,0,5,0,0,1,1,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,0,0,0,0,2,...,0,1,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,2,...,0,0,0,0,0,0,0,0,0,4


### Criando o DataFrame final

Agora vamos juntar os metadados das noticias com as colunas do Bag of Words. Para evitar conflito de nomes, as colunas do Bag of Words recebem o prefixo `bow_`.

In [93]:
metadados = df[["data", "tags", "titulo", "url"]].reset_index(drop=True)
bow_com_prefixo = df_bow.add_prefix("bow_").reset_index(drop=True)

df_final = pd.concat([metadados, bow_com_prefixo], axis=1)

df_final.head()

,data,tags,titulo,url,bow_abertura,bow_abordadas,bow_abraao,bow_abril,bow_abrir,bow_absolvicao,...,bow_votarmos,bow_votos,bow_voz,bow_vulnerabilidades,bow_weverton,bow_wilder,bow_wolney,bow_xingamentos,bow_zequinha,bow_zettel
0,2026-03-30T17:31:28-03:00,"[Agricultura familiar , Emendas parlamentares,...",Chico Rodrigues destaca ações em Roraima e cri...,https://www12.senado.leg.br/noticias/materias/...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2026-03-27T19:39:12-03:00,"[Aposentados, INSS, Pensionistas]","Com mais de 4 mil páginas, relatório da CPMI p...",https://www12.senado.leg.br/noticias/materias/...,0,0,1,0,0,0,...,1,1,2,0,5,0,0,1,1,0
2,2026-03-24T16:59:17-03:00,"[Alexandre de Moraes, Jair Bolsonaro, Procurad...",Wilder defende prisão domiciliar para Jair Bol...,https://www12.senado.leg.br/noticias/materias/...,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,2026-03-26T21:14:18-03:00,"[André Mendonça, Aposentados, Corrupção, INSS,...",CPMI do INSS: relatório será lido e pode ser v...,https://www12.senado.leg.br/noticias/materias/...,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4,2026-03-26T20:05:27-03:00,"[Aposentados, Conselho de Controle de Atividad...",CPMI do INSS aprova mais uma convocação e queb...,https://www12.senado.leg.br/noticias/materias/...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4


## Exemplo de analise

Agora podemos fazer uma analise simples das palavras do Bag of Words: quais aparecem mais, quais aparecem menos e quantas palavras diferentes temos no vocabulario.

In [94]:
colunas_bow = [coluna for coluna in df_final.columns if coluna.startswith("bow_")]

frequencia_palavras = df_final[colunas_bow].sum().sort_values(ascending=False)

print(f"Total de palavras diferentes: {len(frequencia_palavras)}")

frequencia_palavras.head(10)

Total de palavras diferentes: 1681


bow_cpmi           56
bow_senado         52
bow_senador        45
bow_presidente     42
bow_ex             37
bow_ministro       34
bow_agencia        32
bow_inss           31
bow_prorrogacao    31
bow_stf            30
dtype: int64

In [95]:
frequencia_palavras.tail(10)

bow_adequada          1
bow_adentrar          1
bow_acusado           1
bow_acusacao          1
bow_acrescentando     1
bow_acrescenta        1
bow_aconteceu         1
bow_acontecendo       1
bow_acompanhou        1
bow_acompanhamento    1
dtype: int64

In [96]:
documentos_por_palavra = (df_final[colunas_bow] > 0).sum().sort_values(ascending=False)
documentos_por_palavra.index = documentos_por_palavra.index.str.replace("bow_", "", regex=False)

documentos_por_palavra.head(10)

agencia       16
feira         16
reproducao    16
autorizada    16
mediante      16
senado        16
senador       16
citacao       16
nesta         13
presidente    12
dtype: int64

In [97]:
df_final["palavras_unicas"] = (df_final[colunas_bow] > 0).sum(axis=1)

df_final[["titulo", "palavras_unicas"]].sort_values("palavras_unicas", ascending=True).head(10)

,titulo,palavras_unicas
9,Veneziano celebra 60 anos de fundação do MDB,80
8,Cancelada reunião da CPMI do INSS desta quarta...,87
7,Cleitinho defende revogação da condenação de B...,96
12,Girão diz ser vítima de censura e critica Comu...,97
5,Girão pede ao STF a abertura da CPI do Banco M...,118
15,CPI do Crime repudia anulação da quebra de sig...,126
2,Wilder defende prisão domiciliar para Jair Bol...,126
13,Cancelada reunião da CPMI do INSS desta segunda,140
10,Kajuru manifesta apoio à prisão domiciliar par...,154
0,Chico Rodrigues destaca ações em Roraima e cri...,160


## 4. TF-IDF

O Bag of Words transforma textos em numeros contando quantas vezes cada palavra aparece em cada noticia. Essa representacao e simples e util, mas tem uma limitacao importante: palavras muito frequentes podem parecer importantes apenas porque aparecem muito.

O **TF-IDF** e uma forma de dar peso para as palavras tentando responder a duas perguntas ao mesmo tempo:

- Esta palavra aparece bastante nesta noticia?
- Esta palavra e rara no conjunto de noticias?

A ideia e que uma palavra deve receber peso alto quando aparece bastante em uma noticia, mas nao aparece em todas as noticias. Assim, o TF-IDF ajuda a destacar palavras mais caracteristicas de cada documento.

### TF: Term Frequency

**TF** significa *Term Frequency*, ou frequencia do termo.

Ele mede a importancia de uma palavra dentro de uma noticia especifica. Se uma palavra aparece muitas vezes em uma noticia, o TF dela naquela noticia sera maior.

$$TF(t, d) = \frac{\text{quantidade do termo } t \text{ no documento } d}{\text{total de termos no documento } d}$$

Exemplo: se uma noticia tem 100 palavras e a palavra `senado` aparece 5 vezes, entao:

$$TF(\text{senado}) = \frac{5}{100} = 0.05$$

O TF olha apenas para dentro de uma noticia. Ele ainda nao sabe se a palavra e comum ou rara no conjunto inteiro.

### IDF: Inverse Document Frequency

**IDF** significa *Inverse Document Frequency*, ou frequencia inversa nos documentos.

Ele mede o quanto uma palavra e rara no conjunto de noticias. Palavras que aparecem em muitas noticias recebem peso menor. Palavras que aparecem em poucas noticias recebem peso maior.

$$IDF(t) = \log\left(\frac{N}{DF(t)}\right)$$

Onde:

- $N$ e o numero total de noticias.
- $DF(t)$ e o numero de noticias em que o termo $t$ aparece.

Se uma palavra aparece em todas as noticias, ela nao ajuda muito a diferenciar uma noticia da outra. Por isso, seu IDF fica baixo.

### TF-IDF

O **TF-IDF** combina as duas ideias:

$$TF\text{-}IDF(t, d) = TF(t, d) \times IDF(t)$$

Uma palavra tera peso alto quando:

- aparece bastante em uma noticia;
- nao aparece em muitas outras noticias.

Por isso, TF-IDF costuma ser melhor do que uma contagem simples quando queremos comparar documentos, encontrar palavras importantes ou preparar textos para modelos de aprendizado de maquina.

## Como calcular TF-IDF na mao

Como ja criamos o `df_final`, temos uma matriz Bag of Words pronta nas colunas que comecam com `bow_`.

Para calcular o TF-IDF manualmente com pandas, o caminho e:

1. Separar apenas as colunas `bow_` do `df_final`.
2. Remover o prefixo `bow_` dos nomes das colunas, para trabalhar diretamente com as palavras.
3. Calcular o total de palavras de cada noticia somando as linhas da matriz.
4. Calcular o **TF** dividindo cada valor da linha pelo total da propria linha.
5. Calcular em quantas noticias cada palavra aparece.
6. Calcular o **IDF** usando $\log(N / DF)$.
7. Multiplicar a matriz de TF pelo vetor de IDF.

O resultado final e uma nova matriz: cada linha continua sendo uma noticia, cada coluna continua sendo uma palavra, mas agora os valores sao pesos TF-IDF em vez de contagens brutas.

## Implementacao

Agora vamos implementar. A ideia e preencher as proximas celulas seguindo os passos explicados acima.

In [98]:
matriz_bow_df = df_final[colunas_bow].copy()
matriz_bow_df.columns = matriz_bow_df.columns.str.replace("bow_", "", regex=False)

matriz_bow_df.head()

,abertura,abordadas,abraao,abril,abrir,absolvicao,abuso,ac,acabou,acao,...,votarmos,votos,voz,vulnerabilidades,weverton,wilder,wolney,xingamentos,zequinha,zettel
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,0,0,0,1,0,0,...,1,1,2,0,5,0,0,1,1,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,0,0,0,0,2,...,0,1,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,2,...,0,0,0,0,0,0,0,0,0,4


In [99]:
total_palavras_por_noticia = matriz_bow_df.sum(axis=1)

matriz_tf = matriz_bow_df.div(total_palavras_por_noticia, axis=0)

matriz_tf.head()

,abertura,abordadas,abraao,abril,abrir,absolvicao,abuso,ac,acabou,acao,...,votarmos,votos,voz,vulnerabilidades,weverton,wilder,wolney,xingamentos,zequinha,zettel
0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
1,0.0,0.0,0.001044,0.0,0.0,0.0,0.0,0.001044,0.0,0.000000,...,0.001044,0.001044,0.002088,0.0,0.005219,0.000000,0.0,0.001044,0.001044,0.000000
2,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.006211,0.0,0.000000,0.000000,0.000000
3,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.008065,...,0.000000,0.004032,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
4,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.004167,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.008333


In [100]:
noticia = 0

print(df_final.loc[noticia, "titulo"])

matriz_tf.loc[noticia].sort_values(ascending=False).head(10)

Chico Rodrigues destaca ações em Roraima e critica polarização política


agricultura     0.020619
afirmou         0.015464
politica        0.015464
produtores      0.015464
senador         0.015464
familiar        0.015464
dialogo         0.010309
acoes           0.010309
entrega         0.010309
equipamentos    0.010309
Name: 0, dtype: float64

In [101]:
import numpy as np

N = len(matriz_bow_df)
documentos_por_termo = (matriz_bow_df > 0).sum(axis=0)

idf = np.log(N / documentos_por_termo)

df_idf = pd.DataFrame({
    "documentos_com_o_termo": documentos_por_termo,
    "idf": idf
}).sort_values("idf", ascending=False)

df_idf.head(10)

,documentos_com_o_termo,idf
zettel,1,2.772589
voltou,1,2.772589
voltados,1,2.772589
vivo,1,2.772589
vivencia,1,2.772589
vive,1,2.772589
vital,1,2.772589
vista,1,2.772589
visita,1,2.772589
violacoes,1,2.772589


In [102]:
df_idf.tail(10)

,documentos_com_o_termo,idf
presidente,12,0.287682
nesta,13,0.207639
reproducao,16,0.000000
feira,16,0.000000
senado,16,0.000000
senador,16,0.000000
agencia,16,0.000000
citacao,16,0.000000
mediante,16,0.000000
autorizada,16,0.000000


In [103]:
matriz_tfidf = matriz_tf.mul(idf, axis=1)

matriz_tfidf.head()

,abertura,abordadas,abraao,abril,abrir,absolvicao,abuso,ac,acabou,acao,...,votarmos,votos,voz,vulnerabilidades,weverton,wilder,wolney,xingamentos,zequinha,zettel
0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
1,0.0,0.0,0.002894,0.0,0.0,0.0,0.0,0.002894,0.0,0.000000,...,0.002894,0.002171,0.005788,0.0,0.014471,0.000000,0.0,0.002894,0.002894,0.000000
2,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.017221,0.0,0.000000,0.000000,0.000000
3,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.013500,...,0.000000,0.008385,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
4,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.006975,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.023105


In [110]:
noticia = 10

print(df_final.loc[noticia, "titulo"])

matriz_tfidf.loc[noticia].sort_values(ascending=False).head(10)

Kajuru manifesta apoio à prisão domiciliar para Bolsonaro


kajuru        0.044243
domiciliar    0.035617
prisao        0.029496
estado        0.024748
sob           0.022122
concessao     0.022122
medica        0.022122
condicoes     0.022122
ex            0.018435
pessoa        0.017808
Name: 10, dtype: float64

In [111]:
tfidf_com_prefixo = matriz_tfidf.add_prefix("tfidf_").reset_index(drop=True)

df_final_tfidf = pd.concat([metadados, tfidf_com_prefixo], axis=1)

df_final_tfidf.head()

,data,tags,titulo,url,tfidf_abertura,tfidf_abordadas,tfidf_abraao,tfidf_abril,tfidf_abrir,tfidf_absolvicao,...,tfidf_votarmos,tfidf_votos,tfidf_voz,tfidf_vulnerabilidades,tfidf_weverton,tfidf_wilder,tfidf_wolney,tfidf_xingamentos,tfidf_zequinha,tfidf_zettel
0,2026-03-30T17:31:28-03:00,"[Agricultura familiar , Emendas parlamentares,...",Chico Rodrigues destaca ações em Roraima e cri...,https://www12.senado.leg.br/noticias/materias/...,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
1,2026-03-27T19:39:12-03:00,"[Aposentados, INSS, Pensionistas]","Com mais de 4 mil páginas, relatório da CPMI p...",https://www12.senado.leg.br/noticias/materias/...,0.0,0.0,0.002894,0.0,0.0,0.0,...,0.002894,0.002171,0.005788,0.0,0.014471,0.000000,0.0,0.002894,0.002894,0.000000
2,2026-03-24T16:59:17-03:00,"[Alexandre de Moraes, Jair Bolsonaro, Procurad...",Wilder defende prisão domiciliar para Jair Bol...,https://www12.senado.leg.br/noticias/materias/...,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.0,0.000000,0.017221,0.0,0.000000,0.000000,0.000000
3,2026-03-26T21:14:18-03:00,"[André Mendonça, Aposentados, Corrupção, INSS,...",CPMI do INSS: relatório será lido e pode ser v...,https://www12.senado.leg.br/noticias/materias/...,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.000000,0.008385,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
4,2026-03-26T20:05:27-03:00,"[Aposentados, Conselho de Controle de Atividad...",CPMI do INSS aprova mais uma convocação e queb...,https://www12.senado.leg.br/noticias/materias/...,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.023105


## Como calcular TF-IDF com scikit-learn

Na pratica, tambem podemos usar o `TfidfVectorizer` do scikit-learn.

Ele faz em uma unica etapa o que fizemos manualmente:

- cria o vocabulario;
- conta os termos;
- calcula os pesos TF-IDF;
- devolve uma matriz numerica.

Como nossos textos ja foram limpos anteriormente, podemos aplicar o `TfidfVectorizer` diretamente em `df["texto_sem_stopwords"]`.

A vantagem do scikit-learn e que ele ja resolve detalhes de implementacao e devolve uma matriz pronta para ser usada em modelos de aprendizado de maquina. A vantagem de fazer na mao primeiro e entender o que esta acontecendo por baixo.

In [112]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer_tfidf = TfidfVectorizer()
matriz_tfidf_sklearn = vectorizer_tfidf.fit_transform(df["texto_sem_stopwords"])

df_tfidf_sklearn = pd.DataFrame(
    matriz_tfidf_sklearn.toarray(),
    columns=vectorizer_tfidf.get_feature_names_out()
)

df_tfidf_sklearn.head()

,02,04,061,0800,09,11,12,120,16,16h,...,votarmos,votos,voz,vulnerabilidades,weverton,wilder,wolney,xingamentos,zequinha,zettel
0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
1,0.025294,0.025294,0.0,0.0,0.050587,0.025294,0.025294,0.000000,0.050587,0.0,...,0.025294,0.022028,0.050587,0.0,0.126468,0.000000,0.0,0.025294,0.025294,0.000000
2,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.0,0.000000,0.083952,0.0,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.057203,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.034867,0.000000,0.0,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.160147
